## This workbook creates an NVT Gibbs Ensemble Monte Carlo (GEMC-NVT) simulation using a few alkanes, to evaluate the liquid-vapor equilibrium. 

### These file writers utilize the new GOMC file writer via MoSDeF, which incorporates a GOMC force field (CHARMM style force field), psf, and pdb writers.  The main difference in these psf and pdb writers is the ability to actively select properties based on the molecule's residue, such as the force field, and fixing bond lengths and angles.  

### Build the individual molecules and minimize their energies by applying the appropriate force fields. 

In [1]:
import mbuild as mb
from foyer import Forcefield
import mbuild.formats.charmm_writer as mf_charmm

forcefield_names = 'trappe-ua'
FF =  Forcefield(name=forcefield_names)

Molecule_A =mb.load('files/iso_decane.mol2')
FF.apply(Molecule_A)
Molecule_A.name = 'IDE'  # naming the molecule (i.e., for the used shortly residue)

Molecule_B =mb.load('files/iso_octane.mol2')
FF.apply(Molecule_B)
Molecule_B.name = 'IOT'  # naming the molecule (i.e., for the used shortly residue)

Molecule_A.visualize()

/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/foyer/forcefield.py:473: UserWarning: Non-atomistic element type detected. Creating custom element for _CH4
  'Creating custom element for {}'.format(element))
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/foyer/forcefield.py:473: UserWarning: Non-atomistic element type detected. Creating custom element for _CH3
  'Creating custom element for {}'.format(element))
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/foyer/forcefield.py:473: UserWarning: Non-atomistic element type detected. Creating custom element for _CH2
  'Creating custom element for {}'.format(element))
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/foyer/forcefield.py:473: UserWarning: Non-atomistic element type detected. Creating custom element for _HC
  'Creating custom element for {}'.format(element))
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/foyer/force

You appear to be running in JupyterLab (or JavaScript failed to load for some other reason). You need to install the 3dmol extension: 
 jupyter labextension install jupyterlab_3dmol

### Enter the data required to build the liquid and vapor boxes via the GOMC format

In [2]:
approx_total_No_atoms_liq = 10000
approx_total_No_atoms_per_cg_atom = 3.25    # 1 for AA and ~3.0 to 3.5 for UA.
est_beads_per_moledule = 9
est_total_molecules_liq = approx_total_No_atoms_liq / approx_total_No_atoms_per_cg_atom/est_beads_per_moledule

vol_vap_div_vol_liq = 10
molecules_vap_div_molecules_liq = 0.1

mol_fraction_Molecule_A = 0.25
mol_fraction_Molecule_B = 0.75

Molecule_Type_List = [Molecule_A, Molecule_B ]
Molecule_ResName_List = [Molecule_A.name, Molecule_B.name ]
Total_molecules_liquid = [int(mol_fraction_Molecule_A * est_total_molecules_liq) ,
                          int(mol_fraction_Molecule_B * est_total_molecules_liq)]
Total_molecules_vapor = [int(mol_fraction_Molecule_A * est_total_molecules_liq/10) ,
                         int(mol_fraction_Molecule_B * est_total_molecules_liq/10)]

/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


### Choose an atom name for the beads based on preference or main element. Note: a choice of 2 letters or less is preferred so the beads can have greater than 9 unique atom names. 

### If C is chosen to represent the bead, the molecule is labeled C1, C2, C3,..., Cx in sequential order. These unique atom names are critical to using GOMC's special Molecular Exchange Monte Carlo (MEMC) moves, which substantially increase sampling; thus, improving the simulation efficiency.

In [10]:
Bead_to_atom_name_dict = { '_CH3':'C', '_CH2':'C',  '_CH':'C', '_HC':'C'}

/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


### Build the liquid and vapor boxes

In [7]:
print('Running: liquid phase box packing')
box_liq = mb.fill_box(compound=Molecule_Type_List,
                      n_compounds=Total_molecules_liquid,
                      box=[4, 4, 4])
print('Completed: liquid phase box packing')

print('Running: vapor phase box packing')
box_vap = mb.fill_box(compound=Molecule_Type_List,
                      n_compounds=Total_molecules_vapor,
                      box=[8, 8, 8])
print('Completed: vapor phase box packing')

box_liq.visualize()

/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Running: liquid phase box packing
Completed: liquid phase box packing
Running: vapor phase box packing
Completed: vapor phase box packing


/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/conversion.py:972: UserWarning: Guessing that "<_CH3 pos=( 1.7666, 1.4014, 1.8242), 0 bonds, id: 140424440675600>" is element: "EP"
  atom, element))
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/conversion.py:972: UserWarning: Guessing that "<_HC pos=( 1.4046, 3.6315, 0.8985), 0 bonds, id: 140424116103888>" is element: "EP"
  atom, element))


You appear to be running in JupyterLab (or JavaScript failed to load for some other reason). You need to install the 3dmol extension: 
 jupyter labextension install jupyterlab_3dmol

### Write the GOMC FF file and the psf/pdb files.  

### These writers are intentionally combined for the liquid-vapor boxes (or two other boxes) to ensure the proper FFs and atom types are assigned to all files.  Many simulations do not have all the atom types or molecules in all the simulation boxes (Example: Gibbs and grand-canonical ensembles). 

### Note: the extensions of the psf/pdb files ('IOT_IDE_liquid_box' and 'IOT_IDE_vapor_box') are note entered as they are automatically added.

In [12]:
print('Running: GOMC FF file, and the psf and pdb files')
mf_charmm.charmm_psf_psb_FF(box_liq,
                            'IOT_IDE_liquid_box',
                            structure_1 =box_vap ,
                            filename_1 = 'IOT_IDE_vapor_box',
                            GOMC_FF_filename ="GOMC_IOT_IDE_FF.inp" ,
                            forcefield_names= forcefield_names ,
                            residues= Molecule_ResName_List ,
                            Bead_to_atom_name_dict = Bead_to_atom_name_dict,
                            fix_residue = None,
                            fix_res_bonds_angles = None,
                            reorder_res_in_pdb_psf = False)
print('Completed: GOMC FF file, and the psf and pdb files')

/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Running: GOMC FF file, and the psf and pdb files
write_gomcdata: forcefield_names = trappe-ua, residues = ['IDE', 'IOT']
FF forcefield_names = {'IDE': 'trappe-ua', 'IOT': 'trappe-ua'}
******************************

GOMC FF writing each residues FF as a group for structure_0


/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/conversion.py:972: UserWarning: Guessing that "<_CH3 pos=( 1.7666, 1.4014, 1.8242), 0 bonds, id: 140424100968656>" is element: "EP"
  atom, element))
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/conversion.py:972: UserWarning: Guessing that "<_CH3 pos=( 1.1694, 3.3704, 1.0792), 0 bonds, id: 140424109701264>" is element: "EP"
  atom, element))
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/conversion.py:972: UserWarning: Guessing that "<_HC pos=( 1.4046, 3.6315, 0.8985), 0 bonds, id: 140424109702224>" is element: "EP"
  atom, element))


GOMC FF writing each residues FF as a group for  structure_1


/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/conversion.py:972: UserWarning: Guessing that "<_CH3 pos=( 3.0259, 0.7050, 6.6213), 0 bonds, id: 140423349412624>" is element: "EP"
  atom, element))
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/conversion.py:972: UserWarning: Guessing that "<_CH3 pos=( 3.2303, 3.4074, 5.7143), 0 bonds, id: 140423197348496>" is element: "EP"
  atom, element))
/home/brad/Programs/anaconda3/envs/GOMC_0/lib/python3.7/site-packages/mbuild/conversion.py:972: UserWarning: Guessing that "<_HC pos=( 3.1181, 3.0627, 5.8717), 0 bonds, id: 140423197349456>" is element: "EP"
  atom, element))


forcefield type from compound = {'IDE': 'trappe-ua', 'IOT': 'trappe-ua'}
coulomb14scale from compound = {'IDE': 0.0, 'IOT': 0.0}
lj14scale from compound = {'IDE': 0.0, 'IOT': 0.0}
No urey bradley terms detected, will use angle_style harmonic
will use CHARMM_torsions  =  K0 + K1 * (1 + Cos[n1*(t) - (d1)] ) + K2 * (1 + Cos[n2*(t) - (d2)] ) + K3 * (1 + Cos[n3*(t) - (d3)] ) + K4 * (1 + Cos[n4*(t) - (d4)] ) + K5 * (1 + Cos[n5*(t) - (d5)] ) 
******************************

writing the GOMC force field file 
! RB-torsion to CHARMM dihedral conversion error is OK [error <= 10^(-10)]
! Maximum( |(RB-torsion calc)-(CHARMM dihedral calc)| ) =  1.3322676295501878e-14

NBFIX_Mixing not used or no mixing used for the non-bonded potentials out

******************************

write_charmm_psf file is running
write_charmm_psf: forcefield_names = {'IDE': 'trappe-ua', 'IOT': 'trappe-ua'}, residues = ['IDE', 'IOT']
******************************

No urey bradley terms detected
RB Torsions detected, will 